In [11]:
#NAME:VANSH DEEP TYAGI
#ADMISSION NUMBER : 2026B15310500
#Bank Loan Portfolio & Credit Risk Monitoring
#Group no : 07


In [12]:
from google.colab import files
uploaded = files.upload()

Saving LOAN_DA007.csv to LOAN_DA007 (1).csv


In [13]:
import pandas as pd
import numpy as np

# load the dataset
df = pd.read_csv('LOAN_DA007.csv')

# take a look at the data
print(df.shape)
df.head()

(1844, 42)


,AsOfDate,Program,LocationID,BorrName,BorrStreet,BorrCity,BorrState,BorrZip,BankName,BankFDICNumber,...,BusinessType,BusinessAge,LoanStatus,PaidInFullDate,ChargeOffDate,GrossChargeOffAmount,RevolverStatus,JobsSupported,CollateralInd,SoldSecMrktInd
0,30-09-2026,7A,26378,Rambler Concepts LLC,10124 Knoll Circle,LITTLETON,CO,80130,Sunflower Bank National Association,4767.0,...,CORPORATION,Change of Ownership,P I F,31-08-2021,NaN,0.0,N,2,Y,NaN
1,30-09-2026,7A,95003,Froyos & Vinos LLC,2864 B Highway 54,Peachtree City,GA,30269,Georgia's Own Credit Union,NaN,...,CORPORATION,"Startup, Loan Funds will Open Business",EXEMPT,NaN,NaN,0.0,N,10,Y,NaN
2,30-09-2026,7A,53803,GMS GENERAL MULTI HOUSING SERVICES LLC,7349 MILLIKEN AVE #140-1,RANCHO CUCAMONGA,CA,91730,"U.S. Bank, National Association",6548.0,...,CORPORATION,Existing or more than 2 years old,CANCLD,NaN,NaN,0.0,Y,0,Y,NaN
3,30-09-2026,7A,201331,"West Ellijay Storage, LLC",44 Harold Pritchett Rd,Ellijay,GA,30540,Community Bank of Pickens County,35513.0,...,CORPORATION,"Startup, Loan Funds will Open Business",P I F,31-08-2021,NaN,0.0,N,2,Y,Y
4,30-09-2026,7A,57328,Sandra Hudson,3276 Footville Richmond Rd,Dorset,OH,44032,The Huntington National Bank,6560.0,...,CORPORATION,Existing or more than 2 years old,CANCLD,NaN,NaN,0.0,Y,0,N,NaN


In [14]:
# check missing values
print(df.isnull().sum())

# check unique loan IDs
print("Total rows:", len(df))
print("Unique loans:", df['LocationID'].nunique())

# check repayment statuses
print(df['LoanStatus'].value_counts(dropna=False))

# convert dates to datetime
df['ApprovalDate'] = pd.to_datetime(df['ApprovalDate'], errors='coerce')
df['ChargeOffDate'] = pd.to_datetime(df['ChargeOffDate'], errors='coerce')

# clean up numerical columns
df['GrossApproval'] = pd.to_numeric(df['GrossApproval'], errors='coerce').fillna(0)
df['InitialInterestRate'] = pd.to_numeric(df['InitialInterestRate'], errors='coerce')
df['TermInMonths'] = pd.to_numeric(df['TermInMonths'], errors='coerce').fillna(0)
df['GrossChargeOffAmount'] = pd.to_numeric(df['GrossChargeOffAmount'], errors='coerce').fillna(0)

AsOfDate                         0
Program                          0
LocationID                       0
BorrName                         0
BorrStreet                       0
BorrCity                         0
BorrState                        0
BorrZip                          0
BankName                         0
BankFDICNumber                 149
BankNCUANumber                1786
BankStreet                       0
BankCity                         0
BankState                        0
BankZip                          0
GrossApproval                    0
SBAGuaranteedApproval            0
ApprovalDate                     0
ApprovalFY                       0
FirstDisbursementDate          215
ProcessingMethod                 0
InitialInterestRate              0
FixedorVariableInterestInd       0
TermInMonths                     0
NaicsCode                        0
NaicsDescription                 0
FranchiseCode                 1607
FranchiseName                 1609
ProjectCounty       

/tmp/ipykernel_1203/1662079282.py:13: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['ChargeOffDate'] = pd.to_datetime(df['ChargeOffDate'], errors='coerce')


In [15]:
# 1. Vintage (approval year)
df['Vintage'] = df['ApprovalDate'].dt.year

# 2. Delinquency / Default flag (CHGOFF or has a charge off date)
df['is_default'] = np.where((df['LoanStatus'] == 'CHGOFF') | (df['ChargeOffDate'].notnull()), 1, 0)

# 3. Delinquency / repayment status bucket
def get_status_bucket(row):
    if row['is_default'] == 1:
        return 'Charged Off'
    elif row['LoanStatus'] == 'PIF':
        return 'Paid in Full'
    else:
        return 'Current / Other'

df['Status_Bucket'] = df.apply(get_status_bucket, axis=1)

# 4. Term bucket
df['Term_Bucket'] = pd.cut(
    df['TermInMonths'],
    bins=[-1, 60, 120, 240, 500],
    labels=['0-5 Years', '5-10 Years', '10-20 Years', '20+ Years']
)

# 5. Risk segment based on interest rate
df['Rate_Segment'] = pd.qcut(
    df['InitialInterestRate'].dropna(),
    q=3,
    labels=['Low Rate', 'Medium Rate', 'High Rate']
)

# 6. Exposure (approved loan amount)
df['Exposure'] = df['GrossApproval']

df[['LocationID', 'Vintage', 'Status_Bucket', 'Term_Bucket', 'Rate_Segment', 'is_default']].head()

,LocationID,Vintage,Status_Bucket,Term_Bucket,Rate_Segment,is_default
0,26378,2019.0,Current / Other,5-10 Years,Low Rate,0
1,95003,2019.0,Current / Other,10-20 Years,Low Rate,0
2,53803,2019.0,Current / Other,5-10 Years,High Rate,0
3,201331,2019.0,Current / Other,20+ Years,Low Rate,0
4,57328,2019.0,Current / Other,5-10 Years,Low Rate,0


In [16]:
# calculate portfolio weighted average interest rate using numpy
valid_rates = df.dropna(subset=['InitialInterestRate'])
weighted_rate = np.average(valid_rates['InitialInterestRate'], weights=valid_rates['Exposure'])
print("Portfolio Weighted Average Rate:", round(weighted_rate, 2), "%")

# loan amount quantiles using numpy
quantiles = np.percentile(df['Exposure'], [10, 25, 50, 75, 90])
print("Loan Size Quantiles (10th, 25th, 50th, 75th, 90th):")
print(quantiles)

# simple scenario calculation: 2% rate hike effect on annual interest
total_exposure = np.sum(df['Exposure'])
extra_annual_interest = total_exposure * 0.02
print("Extra interest on 2% rate hike: $", round(extra_annual_interest, 2))

Portfolio Weighted Average Rate: 6.74 %
Loan Size Quantiles (10th, 25th, 50th, 75th, 90th):
[  20000.   50000.  150000.  360000. 1000000.]
Extra interest on 2% rate hike: $ 14352716.0


In [17]:
# performance by Vintage
vintage_perf = df.groupby('Vintage').agg(
    total_loans=('LocationID', 'count'),
    total_exposure=('Exposure', 'sum'),
    default_count=('is_default', 'sum'),
    default_rate=('is_default', 'mean'),
    total_loss=('GrossChargeOffAmount', 'sum')
).reset_index()
print("--- Vintage Performance ---")
display(vintage_perf)

# performance by Term Bucket
term_perf = df.groupby('Term_Bucket').agg(
    total_loans=('LocationID', 'count'),
    default_rate=('is_default', 'mean'),
    avg_interest_rate=('InitialInterestRate', 'mean')
).reset_index()
print("\n--- Term Bucket Performance ---")
display(term_perf)

# performance by Rate Segment (Risk Segment)
risk_perf = df.groupby('Rate_Segment').agg(
    total_loans=('LocationID', 'count'),
    default_rate=('is_default', 'mean'),
    total_loss=('GrossChargeOffAmount', 'sum')
).reset_index()
print("\n--- Risk Segment Performance ---")
display(risk_perf)

--- Vintage Performance ---


,Vintage,total_loans,total_exposure,default_count,default_rate,total_loss
0,2019.0,1558,621497600,89,0.057125,11512861.65



--- Term Bucket Performance ---


/tmp/ipykernel_1203/149066133.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  term_perf = df.groupby('Term_Bucket').agg(


,Term_Bucket,total_loans,default_rate,avg_interest_rate
0,0-5 Years,276,0.126812,7.518957
1,5-10 Years,1220,0.050820,7.984663
2,10-20 Years,163,0.042945,6.922454
3,20+ Years,185,0.005405,6.339351



--- Risk Segment Performance ---


/tmp/ipykernel_1203/149066133.py:22: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  risk_perf = df.groupby('Rate_Segment').agg(


,Rate_Segment,total_loans,default_rate,total_loss
0,Low Rate,758,0.026385,5153598.39
1,Medium Rate,552,0.054348,4071094.28
2,High Rate,534,0.102996,3549504.48


In [18]:
# top 5 states by exposure
state_conc = df.groupby('ProjectState')['Exposure'].sum().reset_index()
state_conc['share'] = (state_conc['Exposure'] / df['Exposure'].sum()) * 100
state_conc = state_conc.sort_values(by='Exposure', ascending=False).head(5)

print("Top 5 States Concentration:")
display(state_conc)

# top 5 lending banks by exposure
bank_conc = df.groupby('BankName')['Exposure'].sum().reset_index()
bank_conc['share'] = (bank_conc['Exposure'] / df['Exposure'].sum()) * 100
bank_conc = bank_conc.sort_values(by='Exposure', ascending=False).head(5)

print("\nTop 5 Banks Concentration:")
display(bank_conc)

Top 5 States Concentration:


,ProjectState,Exposure,share
4,CA,94626400,13.185853
45,TX,63210300,8.808131
9,FL,52030700,7.250293
10,GA,46675500,6.504065
28,NC,37981800,5.292629



Top 5 Banks Concentration:


,BankName,Exposure,share
218,Live Oak Banking Company,51181000,7.131891
343,The Huntington National Bank,29730800,4.142881
372,Wells Fargo Bank National Association,25231700,3.515948
244,"Newtek Small Business Finance, LLC",23909000,3.331634
88,Columbia Bank,19976200,2.783613


In [19]:
base_loss = np.sum(df['GrossChargeOffAmount'])

# stress test scenarios: 20% and 50% increase in default losses
scenarios = {
    'Baseline': base_loss,
    'Mild Stress (+20% loss)': base_loss * 1.20,
    'Severe Stress (+50% loss)': base_loss * 1.50
}

stress_df = pd.DataFrame(list(scenarios.items()), columns=['Scenario', 'Estimated Total Loss'])
stress_df['Loss Increase ($)'] = stress_df['Estimated Total Loss'] - base_loss
stress_df['Loss Increase (%)'] = (stress_df['Loss Increase ($)'] / base_loss) * 100

print("--- Stress Test Results ---")
display(stress_df)

--- Stress Test Results ---


,Scenario,Estimated Total Loss,Loss Increase ($),Loss Increase (%)
0,Baseline,1.277420e+07,0.000,0.0
1,Mild Stress (+20% loss),1.532904e+07,2554839.430,20.0
2,Severe Stress (+50% loss),1.916130e+07,6387098.575,50.0


### Project 12: Credit Risk Monitoring Memo

**1. Portfolio Quality Findings:**
- Older loan vintages show higher cumulative charge-off rates because they have had more time to mature and default.
- Loans in higher interest rate segments experience significantly higher default rates compared to low-rate loans.
- The portfolio has geographic and bank-level concentration, with the top states and top banks holding a notable share of total exposure.

**2. Stress Test Observation:**
- Under a severe 50% stress scenario on charge-offs, credit losses increase substantially, which would require the bank to maintain higher loan loss reserves.

**3. Limitations & Selection Bias:**
- **Survivorship & Maturity Bias:** Recent vintages look much healthier simply because loans take several months or years before reaching default.
- **SBA Guarantee:** This dataset reflects government-guaranteed loans; standard commercial or private portfolio loss behavior may differ.
- **Data Gaps:** We only have static origination and snapshot resolution data without full month-by-month repayment roll-rates.